# AI-Driven Phishing Email Detection using NLP and Hybrid Features
This notebook provides a complete, self-contained pipeline for data preprocessing, advanced hybrid feature engineering, model training/hyperparameter optimization, validation, and inference testing.

## Step 1: Imports and Helper Utilities
We start by importing standard libraries and implementing the feature extraction classes so the notebook is fully autonomous.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin

sns.set_theme(style="whitegrid")

# Preprocessing and Feature Extraction Resources
STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", 
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", 
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", 
    "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", 
    "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", 
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", 
    "with", "about", "against", "between", "into", "through", "during", "before", "after", 
    "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", 
    "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", 
    "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", 
    "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", 
    "should", "now"
}

BRANDS = ["microsoft", "amazon", "google", "paypal", "apple", "facebook", "netflix", "dhl", "fedex", "linkedin"]
SUSPICIOUS_TLDS = {"zip", "mov", "ru", "xyz", "top", "support", "info", "cc", "tk", "gq", "cf", "ml"}
SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly", "rebrand.ly"}
MFA_LURES = {"mfa", "2fa", "otp", "authenticator", "verification code", "one-time", "passcode"}
URGENCY_KEYWORDS = {
    'urgent', 'suspend', 'verify', 'action', 'alert', 'immediately', 'compromised', 'claim', 
    'restricted', 'security', 'update', 'password', 'confirm', 'attention', 'required', 'login',
    'unusual', 'activity', 'invoice', 'overdue', 'billing', 'delivery', 'fedex', 'ups', 'paypal', 
    'crypto', 'wallet', 'authorize', 'deactivate', 'block'
}
IMPERATIVE_VERBS = {"click", "verify", "log", "update", "check", "confirm", "respond", "pay", "download", "open"}

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    tokens = text.split()
    return " ".join([word for word in tokens if word not in STOPWORDS])

def calculate_entropy(s):
    if not s:
        return 0.0
    probabilities = [float(s.count(c)) / len(s) for c in dict.fromkeys(s)]
    entropy = - sum([p * np.log2(p) for p in probabilities])
    return entropy

def find_domain_similarity(domain, brands=BRANDS):
    domain = domain.lower()
    parts = domain.split('.')
    if not parts:
        return 0.0
    name = parts[0]
    similarities = []
    for brand in brands:
        if name == brand:
            similarities.append(1.0)
            continue
        intersection = len(set(name) & set(brand))
        union = len(set(name) | set(brand))
        jaccard = intersection / union if union > 0 else 0.0
        
        if len(name) > 0 and len(brand) > 0:
            subbed = name.replace('1', 'l').replace('0', 'o').replace('rn', 'm').replace('vv', 'w').replace('I', 'l')
            if subbed == brand:
                similarities.append(0.95)
                continue
        similarities.append(jaccard * 0.8)
    return max(similarities) if similarities else 0.0

class EmailFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        features_list = []
        for text in X:
            if not isinstance(text, str):
                text = ""
            length = len(text)
            cap_ratio = sum(1 for c in text if c.isupper()) / (length + 1)
            exclamations = text.count('!')
            money_chars = text.count('$') + text.count('€') + text.count('£') + text.lower().count('usd') + text.lower().count('transfer')
            
            has_spf = 1 if re.search(r'Received-SPF:\s*pass|spf=pass', text, re.IGNORECASE) else 0
            has_dkim = 1 if re.search(r'dkim=pass|DKIM-Signature', text, re.IGNORECASE) else 0
            has_dmarc = 1 if re.search(r'dmarc=pass', text, re.IGNORECASE) else 0
            
            reply_to_match = re.search(r'Reply-To:\s*([^\s@]+@[^\s@>]+)', text, re.IGNORECASE)
            from_match = re.search(r'From:\s*(?:[^<]*<)?([^\s@]+@[^\s@>]+)', text, re.IGNORECASE)
            reply_to_mismatch = 0
            return_path_mismatch = 0
            sender_spoofing = 0
            
            if reply_to_match and from_match:
                from_email = from_match.group(1).lower()
                reply_email = reply_to_match.group(1).lower()
                from_domain = from_email.split('@')[-1]
                reply_domain = reply_email.split('@')[-1]
                if from_domain != reply_domain:
                    reply_to_mismatch = 1
            
            return_path_match = re.search(r'Return-Path:\s*([^\s@]+@[^\s@>]+)', text, re.IGNORECASE)
            if return_path_match and from_match:
                from_email = from_match.group(1).lower()
                ret_email = return_path_match.group(1).lower()
                from_domain = from_email.split('@')[-1]
                ret_domain = ret_email.split('@')[-1]
                if from_domain != ret_domain:
                    return_path_mismatch = 1
                    
            if from_match:
                from_full = re.search(r'From:\s*([^\n]+)', text, re.IGNORECASE)
                if from_full:
                    from_full_str = from_full.group(1).lower()
                    for brand in BRANDS:
                        if brand in from_full_str and brand not in from_match.group(1).lower():
                            sender_spoofing = 1
                            
            if "reply-to" in text.lower() and reply_to_mismatch == 0:
                emails_found = re.findall(r'[a-zA-Z0-9.-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+', text)
                if len(set(emails_found)) > 1:
                    reply_to_mismatch = 1
            
            domain_age = 3650
            if sender_spoofing or reply_to_mismatch or return_path_mismatch:
                domain_age = 15
            elif any(tld in text.lower() for tld in SUSPICIOUS_TLDS):
                domain_age = 90
            
            urls = re.findall(r'(https?://\S+|www\.\S+)', text, re.IGNORECASE)
            url_count = len(urls)
            url_entropy_list = [calculate_entropy(u) for u in urls]
            avg_url_entropy = np.mean(url_entropy_list) if url_entropy_list else 0.0
            redirect_count = sum(1 for u in urls if "redirect" in u.lower() or "forward" in u.lower() or "goto" in u.lower())
            https_count = sum(1 for u in urls if u.lower().startswith("https"))
            https_ratio = https_count / (url_count + 1)
            ip_url_count = sum(1 for u in urls if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', u))
            punycode_count = sum(1 for u in urls if "xn--" in u.lower())
            
            shortener_count = sum(1 for u in urls if any(sh in u.lower() for sh in SHORTENERS))
            suspicious_tld_url_count = sum(1 for u in urls if any(f".{tld}" in u.lower() for tld in SUSPICIOUS_TLDS))
            
            max_similarity = 0.0
            for u in urls:
                domain_part = re.search(r'https?://(?:www\.)?([^/]+)', u, re.IGNORECASE)
                if domain_part:
                    sim = find_domain_similarity(domain_part.group(1))
                    if sim > max_similarity:
                        max_similarity = sim
            
            pos_words = {"please", "thank", "agree", "appreciate", "welcome", "dear", "hello", "regards"}
            neg_words = {"alert", "suspend", "compromise", "violate", "unauthorized", "fail", "error", "warning", "critical"}
            text_lower = text.lower()
            tokens_all = text_lower.split()
            pos_count = sum(1 for t in tokens_all if t in pos_words)
            neg_count = sum(1 for t in tokens_all if t in neg_words)
            sentiment = (pos_count - neg_count) / (len(tokens_all) + 1)
            
            urgency_score = sum(1 for t in tokens_all if t in URGENCY_KEYWORDS)
            emotion_urgency = 1.0 if urgency_score > 2 or "immediately" in text_lower or "urgent" in text_lower else 0.0
            word_lengths = [len(w) for w in tokens_all]
            avg_word_len = np.mean(word_lengths) if word_lengths else 0.0
            imperative_score = sum(1 for t in tokens_all if t in IMPERATIVE_VERBS)
            imperative_ratio = imperative_score / (len(tokens_all) + 1)
            
            pronouns = {"i", "you", "he", "she", "we", "they", "me", "us", "them"}
            pronoun_count = sum(1 for t in tokens_all if t in pronouns)
            stylometric = pronoun_count / (len(tokens_all) + 1)
            brand_detected = 1 if any(brand in text_lower for brand in BRANDS) else 0
            non_standard = sum(1 for t in tokens_all if re.search(r'\d', t) and re.search(r'[a-zA-Z]', t))
            grammar_quality = 1.0 - (non_standard / (len(tokens_all) + 1))
            
            features_list.append({
                "url_count": float(url_count),
                "has_suspicious_tld": float(1 if suspicious_tld_url_count > 0 or any(f".{tld}" in text.lower() for tld in SUSPICIOUS_TLDS) else 0),
                "has_mfa_lure": float(1 if any(word in text_lower for word in MFA_LURES) else 0),
                "urgency_count": float(urgency_score),
                "email_length": float(length),
                "exclamation_count": float(exclamations),
                "money_char_count": float(money_chars),
                "has_spf": float(has_spf),
                "has_dkim": float(has_dkim),
                "has_dmarc": float(has_dmarc),
                "reply_to_mismatch": float(reply_to_mismatch),
                "return_path_mismatch": float(return_path_mismatch),
                "domain_age_days": float(domain_age),
                "sender_spoofing": float(sender_spoofing),
                "url_entropy": float(avg_url_entropy),
                "url_redirects_count": float(redirect_count),
                "https_ratio": float(https_ratio),
                "ip_url_count": float(ip_url_count),
                "punycode_count": float(punycode_count),
                "shortened_url_count": float(shortener_count),
                "domain_similarity_score": float(max_similarity),
                "sentiment_score": float(sentiment),
                "emotion_urgency": float(emotion_urgency),
                "readability_score": float(avg_word_len),
                "capital_ratio": float(cap_ratio),
                "grammar_quality": float(grammar_quality),
                "stylometric_features": float(stylometric),
                "imperative_ratio": float(imperative_ratio),
                "brand_detected": float(brand_detected)
            })
        return pd.DataFrame(features_list)


## Step 2: Data Ingestion & Target Label Mapping
We load `Phishing_Email.csv`, clean the missing records, map binary categories, and sample/stratify the records.

In [ ]:
dataset_path = "Phishing_Email.csv"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Please make sure '{dataset_path}' is placed in the active folder!")

df = pd.read_csv(dataset_path, usecols=["Email Text", "Email Type"])
df = df.dropna(subset=["Email Text", "Email Type"])

# Set labels
df['label'] = df['Email Type'].apply(lambda x: 1 if "phishing" in str(x).lower() else 0)

# Perform stratified downsampling for performance
SAMPLE_SIZE = min(15000, len(df))
if len(df) > SAMPLE_SIZE:
    _, df = train_test_split(df, test_size=SAMPLE_SIZE, stratify=df['label'], random_state=42)

print(f"Loaded {len(df)} total emails.")
print(df['Email Type'].value_counts())


## Step 3: Hybrid Feature Engineering & NLP Vectorization

In [ ]:
extractor = EmailFeatureExtractor()
print("Extracting structure and metadata heuristics...")
meta_df = extractor.transform(df['Email Text'].tolist())

print("Preprocessing text payloads...")
df['cleaned_text'] = df['Email Text'].apply(preprocess_text)

print("Fitting TF-IDF Vectorizer (Max 4000 features)... ")
vectorizer = TfidfVectorizer(max_features=4000)
tfidf_features = vectorizer.fit_transform(df['cleaned_text']).toarray()
tfidf_df = pd.DataFrame(tfidf_features, columns=vectorizer.get_feature_names_out())

print("Normalizing numeric heuristics via MinMaxScaler...")
scaler = MinMaxScaler()
meta_scaled = pd.DataFrame(scaler.fit_transform(meta_df), columns=meta_df.columns)

# Combine and split
X = pd.concat([tfidf_df, meta_scaled], axis=1)
y = df['label'].reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Feature matrix shape: {X.shape}")

## Step 4: Model Tuning & Training
Optimize a Random Forest classifier and train a comparative suite of production models.

In [ ]:
print("Tuning Random Forest model parameters...")
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [15, None],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_param_grid, cv=3, scoring='f1', n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print(f"Best Random Forest parameters found: {rf_grid.best_params_}")

print("Calibrating Random Forest classifier probabilities...")
calibrated_rf = CalibratedClassifierCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, **rf_grid.best_params_), 
    method='sigmoid', cv=3
)
calibrated_rf.fit(X_train, y_train)

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Random Forest (Tuned)": calibrated_rf,
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=200, early_stopping=True, random_state=42)
}

for name, clf in models.items():
    if name != "Random Forest (Tuned)":
        print(f"Training {name}...")
        clf.fit(X_train, y_train)
print("All classifiers trained successfully!")

## Step 5: Model Evaluation & Visualizations
Compare classification metrics, plot the Confusion Matrix and output the Receiver Operating Characteristic (ROC) curve.

In [ ]:
best_model_name = ""
best_f1 = -1
predictions_dict = {}

for name, clf in models.items():
    preds = clf.predict(X_test)
    predictions_dict[name] = preds
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    f1 = report['macro avg']['f1-score']
    print(f"Model: {name}")
    print(f"Accuracy: {report['accuracy']:.4f} | Precision: {report['1']['precision']:.4f} | Recall: {report['1']['recall']:.4f} | F1-Score: {report['1']['f1-score']:.4f}")
    print("-" * 70)
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name

print(f"Best Model Selected: {best_model_name}")
best_clf = models[best_model_name]

# Confusion Matrix Plot
cm = confusion_matrix(y_test, predictions_dict[best_model_name])
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Safe Email", "Phishing Email"], 
            yticklabels=["Safe Email", "Phishing Email"])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ROC Curve Plot
plt.figure(figsize=(8, 6))
for name, clf in models.items():
    if hasattr(clf, "predict_proba"):
        probs = clf.predict_proba(X_test)[:, 1]
    else:
        probs = clf.decision_function(X_test)
        probs = (probs - probs.min()) / (probs.max() - probs.min())
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_score = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Step 6: Model Export / Persistence
Persist serialized representations of the best model, vectorizer, and scaler.

In [ ]:
joblib.dump(best_clf, "best_phishing_model.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(scaler, "metadata_scaler.joblib")

print("Model components saved as:")
print(" - best_phishing_model.joblib")
print(" - tfidf_vectorizer.joblib")
print(" - metadata_scaler.joblib")

## Step 7: Live Inference Testing
Use our custom feature extractor and the trained model pipeline on new, simulated email bodies.

In [ ]:
def predict_email(raw_email):
    cleaned_text = preprocess_text(raw_email)
    
    # Extract structured features
    meta_df = extractor.transform([raw_email])
    meta_scaled = pd.DataFrame(scaler.transform(meta_df), columns=meta_df.columns)
    
    # Vectorize text features
    tfidf_feat = vectorizer.transform([cleaned_text]).toarray()
    tfidf_df = pd.DataFrame(tfidf_feat, columns=vectorizer.get_feature_names_out())
    
    # Combine all features
    X_pred = pd.concat([tfidf_df, meta_scaled], axis=1)
    
    pred_label = best_clf.predict(X_pred)[0]
    pred_prob = best_clf.predict_proba(X_pred)[0][1] if hasattr(best_clf, "predict_proba") else None
    
    return pred_label, pred_prob

test_cases = [
    {
        "desc": "Simulated Urgent Security MFA Bypass Attack",
        "text": "Microsoft Security Alert: Unusual login activity has been detected on your Office365 account from a foreign IP address. To prevent permanent deactivation, you must verify your profile details immediately. Click here to confirm your 2FA Authentication Code: http://verification-mfa-security.support/auth-login"
    },
    {
        "desc": "Simulated Urgent Invoice Billing Scam",
        "text": "Dear customer, your payment of $1,450.00 for the outstanding invoice #94022 is past due. To avoid late service interruption fees, verify details and settle your balance immediately by visiting our secure transfer portal: http://bit.ly/payment-invoice-process"
    },
    {
        "desc": "Legitimate Personal Check-in",
        "text": "Hey there! Long time no see. Let me know if you are free to grab lunch sometime next week. I wanted to catch up and show you the new project I've been working on. Talk soon."
    }
]

print("=== Inference Test Run ===\n")
for idx, case in enumerate(test_cases, start=1):
    label, prob = predict_email(case["text"])
    status = "PHISHING DETECTED" if label == 1 else "SAFE/LEGITIMATE"
    confidence = f" (Threat confidence: {prob*100:.2f}%)" if prob is not None else ""
    print(f"Test Case {idx}: {case['desc']}")
    print(f"Prediction: {status}{confidence}")
    print(f"Email Content: '{case['text']}'")
    print("-" * 70)
